# paper_riskDD -- 03: Cognitive Model & Convergent Evidence

Results section structure:

3. **Cognitive model results** -- bauer risk models; group differences in prior_sd / evidence_sd / RNP
4. **Convergent evidence** -- eyetracking by group; correlations between probit/cognitive model parameters and companion measures (neural precision, NPC dispersion, Corsi span, math anxiety, DEMAT)

**Depends on outputs from NB02:**
- `results/probit_sym_subwise_ss_indpoint_slope.csv`
- `results/probit_nonsym_subwise_ss_indpoint_slope.csv`

**Output files**
- `results/cogmodel_posterior_effects.csv`
- `results/convergent_correlations.csv`
- `figures/` -- paper figures

In [ ]:
import sys, os, os.path as op
import itertools
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import arviz as az

sys.path.insert(0, os.getcwd())
from utils_riskBehav import summarize_posterior, save_posterior_effects

try:
    import pingouin as pg
    HAS_PINGOUIN = True
except ImportError:
    HAS_PINGOUIN = False
    from scipy import stats as ss_stat
    print('pingouin not found -- using scipy Pearson')

sns.set_theme('paper', 'white', font='helvetica', font_scale=1.2)
pal = {'Control': sns.color_palette()[0], 'Dyscalculic': sns.color_palette()[1]}

bids_folder      = '/Users/mrenke/data/ds-dnumrisk'
trace_folder     = op.join(bids_folder, 'derivatives', 'cogmodels_risk')
phenotype_folder = op.join(bids_folder, 'derivatives', 'phenotype')
out_folder       = op.join(bids_folder, 'plots_and_ims', 'paper_riskDD')
figures_folder   = op.join(out_folder, 'figures')
results_folder   = op.join(out_folder, 'results')
for d in [figures_folder, results_folder]:
    os.makedirs(d, exist_ok=True)

EXCLUDED = [32, 40, 45, 46, 50]
print('Setup complete.')

In [ ]:
from numrisk.behavior_risk.utils import get_data

df = get_data()
groupList = df[['group']].groupby('subject').first().astype(int)
groupList['group_label'] = groupList['group'].map({0: 'Control', 1: 'Dyscalculic'})
groupList = groupList[~groupList.index.isin(EXCLUDED)]

# subwise slopes from NB02
df_ss_sym    = pd.read_csv(op.join(results_folder, 'probit_sym_subwise_ss_indpoint_slope.csv'),    index_col='subject')
df_ss_nonsym = pd.read_csv(op.join(results_folder, 'probit_nonsym_subwise_ss_indpoint_slope.csv'), index_col='subject')

print(f'N included: {len(groupList)}')
print(groupList['group_label'].value_counts())

---
## 3. Cognitive Model Results

Models (bauer package): `risk`, `risk_regression`, `power`, `power_regression`.

Available traces:
- `model-risk_rem-32-40-45-46-50_format-symbolic_trace.netcdf`
- `model-risk_regression_rem-32-40-45-46-50_format-symbolic_trace.netcdf`

**Planned analyses:**
1. Posterior distributions of `prior_sd`, `evidence_sd_n1`, `evidence_sd_n2` by group
2. Derived RNP = `get_rnp(evidence_sd, prior_sd)` by group
3. Group differences in cognitive parameters (Bayesian contrasts)
4. Posterior predictive checks

In [ ]:
# TODO: load cognitive model traces
# from numrisk.behavior_risk.utils_02 import build_model, get_rnp
# import bauer
#
# cog_trace = az.from_netcdf(
#     op.join(trace_folder,
#             'model-risk_regression_rem-32-40-45-46-50_format-symbolic_trace.netcdf')
# )

print('Cognitive model section: placeholder -- not yet implemented.')

---
## 4. Convergent Evidence

### 4.1 Eyetracking -- duration difference by group

In [ ]:
# TODO: confirm index column name in eyetracking TSV files before running
# for fmt in ['non-symbolic', 'symbolic']:
#     fn = op.join(phenotype_folder, f'subwise_duration_option_difference_abs_{fmt}.tsv')
#     ...
print('Eyetracking section: placeholder -- check TSV index column name first.')

### 4.2 Correlations: probit ind_point slope x companion measures

Add companion-measure file paths in `companion_files` below when available.

Candidate measures: neural precision (nPRF sigma), NPC dispersion, Corsi span, math anxiety, DEMAT.

In [ ]:
# Primary predictors: ind_point x stake-size slopes from probit models (NB02)
df_corr = df_ss_sym.rename(columns={'ss_slope': 'sym_ss_indpoint_slope'})[['sym_ss_indpoint_slope']]
df_corr = df_corr.join(
    df_ss_nonsym.rename(columns={'ss_slope': 'ns_ss_indpoint_slope'})[['ns_ss_indpoint_slope']],
    how='left'
)
df_corr = df_corr.join(groupList['group_label'])

# Add companion measures here
companion_files = {
    # 'neural_precision': op.join(phenotype_folder, '...csv'),
    # 'npc_dispersion':   op.join(phenotype_folder, '...csv'),
    # 'corsi_span':       op.join(bids_folder, 'add_tables', '...csv'),
    # 'math_anxiety':     op.join(bids_folder, 'add_tables', '...csv'),
    # 'demat':            op.join(bids_folder, 'add_tables', '...csv'),
}
for name, fpath in companion_files.items():
    if op.exists(fpath):
        df_corr = df_corr.join(pd.read_csv(fpath, index_col='subject')[[name]], how='left')
    else:
        print(f'[MISSING] {name}: {fpath}')

print('Correlation dataframe shape:', df_corr.shape)
print(df_corr.head())

In [ ]:
numeric_cols = [c for c in df_corr.columns if c != 'group_label']

if len(numeric_cols) < 2:
    print('Need >=2 numeric columns -- add companion measures above.')
else:
    corr_rows = []
    for (x_var, y_var), grp_label in itertools.product(
        itertools.combinations(numeric_cols, 2),
        df_corr['group_label'].unique()
    ):
        sub_data = df_corr[df_corr['group_label'] == grp_label][[x_var, y_var]].dropna()
        if len(sub_data) < 5:
            continue
        if HAS_PINGOUIN:
            res = pg.corr(sub_data[x_var], sub_data[y_var], method='shepherd')
            corr_rows.append(dict(group=grp_label, x=x_var, y=y_var,
                                  r=res['r'].iloc[0], p=res['p-val'].iloc[0], n=res['n'].iloc[0]))
        else:
            r, p = ss_stat.pearsonr(sub_data[x_var], sub_data[y_var])
            corr_rows.append(dict(group=grp_label, x=x_var, y=y_var, r=r, p=p, n=len(sub_data)))

    if corr_rows:
        df_corr_table = pd.DataFrame(corr_rows)
        df_corr_table.to_csv(op.join(results_folder, 'convergent_correlations.csv'), index=False)
        print('=== Correlations ===')
        print(df_corr_table.round(3).to_string(index=False))
    else:
        print('No correlations computed -- check data.')

### 4.3 Scatter plots for nominally significant correlations (p < 0.05)

In [ ]:
if len(numeric_cols) >= 2 and 'corr_rows' in dir() and corr_rows:
    sig_pairs = list({(r['x'], r['y']) for r in corr_rows if r['p'] < 0.05})
    if sig_pairs:
        for x_var, y_var in sig_pairs:
            tmp = df_corr[[x_var, y_var, 'group_label']].dropna()
            g   = sns.lmplot(data=tmp.reset_index(), x=x_var, y=y_var,
                             hue='group_label', palette=pal, height=3.5, aspect=1.1)
            g.figure.suptitle(f'{x_var} x {y_var}', y=1.01)
            sns.despine()
            g.figure.savefig(op.join(figures_folder, f'corr_{x_var}_{y_var}.pdf'), bbox_inches='tight')
            plt.show()
    else:
        print('No significant correlations to plot.')
else:
    print('Scatter plots: add companion measures in section 4.2 first.')